<a name="top" id="top"></a>

<div align="center">
    <h1>Stochastic Optimization</h1>
    <a href="https://github.com/sa1K">Sai Karthik</a>
    <br>
    <i>Weldon School of Biomedical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/parkyr">Yirang Park</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/bernalde">David E. Bernal Neira</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://colab.research.google.com/github/SECQUOIA/Pharma-optimization-flowsheet/blob/main/Enhanced_model/Pharma_Optimizer.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
    <a href="https://secquoia.github.io/">
        <img src="https://img.shields.io/badge/🌲⚛️🌐-SECQUOIA-blue" alt="SECQUOIA"/>
    </a>
</div>

# Introduction

This notebook extends the deterministic model to handle **demand uncertainty** using two-stage stochastic programming.

## Two-Stage Stochastic Programming Framework

**First Stage (Here-and-Now Decisions):**
- Vendor selection at each manufacturing step: $x_{s,v}$
- Transport route selection: $y_{route}$

**Second Stage (Wait-and-See Decisions):**
- Unmet demand (shortage) per scenario: $u_{\omega}$

**Objective:** Minimize expected total cost across all demand scenarios:
$$\min \sum_{\omega \in \Omega} p_\omega \left[ \text{Production Cost} + \text{Transport Cost} + \text{Penalty} \cdot u_\omega \right]$$

# Setup and Imports

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Install dependencies if in Colab
if IN_COLAB:
    !pip install -q pyomo
    !apt-get install -y -qq glpk-utils
    !pip install highspy
    !sudo apt-get install graphviz graphviz-dev
    !pip install networkx

In [ ]:
import pyomo.environ as pyo
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Import from our package
from pharma_optimizer import (
    Generator,
    DemandScenarioGenerator,
    StochasticProductionOptimizer,
    visualize_solution_from_optimizer
)

# Data Generation

Generate random manufacturing and transportation data.

In [ ]:
# Generate a 3-step, 3-vendor instance
generator = Generator(steps=3, options=3)
generator.createManufacturingData()
generator.createTransportData()

# Display data
prod_df = pd.read_csv("manufacturingData2.csv")
print("Production Data:")
print(prod_df)

# Demand Scenario Generation

The `DemandScenarioGenerator` class creates demand scenarios using:
1. **Discrete scenarios**: Low/Base/High demand levels with specified probabilities
2. **Truncated Normal**: Sampled scenarios from a truncated normal distribution

In [ ]:
# Create scenario generator with mean demand = 100, CV = 20%
scenario_gen = DemandScenarioGenerator(mean_demand=100, cv=0.2, seed=42)

# Generate discrete Low/Base/High scenarios
discrete_scenarios = scenario_gen.generate_discrete_scenarios(
    levels=[0.8, 1.0, 1.3],
    probabilities=[0.2, 0.5, 0.3]
)

print("Discrete Scenarios:")
scenario_gen.summary(discrete_scenarios)

In [ ]:
# Generate scenarios from truncated normal distribution
normal_scenarios = scenario_gen.generate_normal_scenarios(num_scenarios=5)

print("\nTruncated Normal Scenarios:")
scenario_gen.summary(normal_scenarios)

# Stochastic Optimization

Run the two-stage stochastic optimization with demand uncertainty.

In [ ]:
# Create stochastic optimizer with discrete scenarios
stochastic_optimizer = StochasticProductionOptimizer(
    prod_cost_csv="manufacturingData2.csv",
    transport_cost_csv="transportData2.csv",
    scenarios=discrete_scenarios,
    shortage_penalty=1000.0
)

# Solve
result = stochastic_optimizer.solve(solver_name='glpk')
print(f"Solver status: {result.solver.termination_condition}\n")

# Display results
stochastic_optimizer.summary()

# Visualization

Visualize the optimal vendor selection path.

In [ ]:
# Visualize the stochastic solution
G, pos, chosen_path = visualize_solution_from_optimizer(
    stochastic_optimizer,
    title="Stochastic Optimization Solution (Demand Uncertainty)"
)
plt.show()

# Scenario Analysis

Detailed analysis of results across different demand scenarios.

In [ ]:
# Get detailed scenario results
scenario_df = stochastic_optimizer.get_scenario_results()
print("Scenario Results:")
print(scenario_df.to_string(index=False))

# Cost statistics
stats = stochastic_optimizer.get_cost_statistics()
print(f"\nExpected Cost: ${stats['expected_cost']:,.2f}")
print(f"Std Deviation: ${stats['std_dev']:,.2f}")
print(f"Cost Range: ${stats['min_cost']:,.2f} - ${stats['max_cost']:,.2f}")

In [ ]:
# Visualize cost breakdown by scenario
fig, ax = plt.subplots(figsize=(10, 6))

scenarios = scenario_df['scenario'].tolist()
x = np.arange(len(scenarios))
width = 0.25

ax.bar(x - width, scenario_df['prod_cost'], width, label='Production Cost')
ax.bar(x, scenario_df['transport_cost'], width, label='Transport Cost')
ax.bar(x + width, scenario_df['shortage_cost'], width, label='Shortage Cost')

ax.set_xlabel('Scenario', fontsize=12)
ax.set_ylabel('Cost ($)', fontsize=12)
ax.set_title('Cost Breakdown by Scenario', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend()
plt.tight_layout()
plt.show()

# Comparison: Discrete vs. Normal Scenarios

In [ ]:
# Run with normal scenarios
stochastic_normal = StochasticProductionOptimizer(
    prod_cost_csv="manufacturingData2.csv",
    transport_cost_csv="transportData2.csv",
    scenarios=normal_scenarios,
    shortage_penalty=1000.0
)

result = stochastic_normal.solve(solver_name='glpk')

print("=" * 60)
print("COMPARISON: Discrete vs Normal Scenarios")
print("=" * 60)

discrete_stats = stochastic_optimizer.get_cost_statistics()
normal_stats = stochastic_normal.get_cost_statistics()

print(f"\nDiscrete Scenarios (3 scenarios):")
print(f"  Expected Cost: ${discrete_stats['expected_cost']:,.2f}")
print(f"  Std Dev:       ${discrete_stats['std_dev']:,.2f}")

print(f"\nNormal Scenarios (5 scenarios):")
print(f"  Expected Cost: ${normal_stats['expected_cost']:,.2f}")
print(f"  Std Dev:       ${normal_stats['std_dev']:,.2f}")

# Sensitivity Analysis: Shortage Penalty

In [ ]:
# Test different shortage penalty values
penalties = [100, 500, 1000, 2000, 5000]
expected_costs = []

for penalty in penalties:
    optimizer = StochasticProductionOptimizer(
        prod_cost_csv="manufacturingData2.csv",
        transport_cost_csv="transportData2.csv",
        scenarios=discrete_scenarios,
        shortage_penalty=penalty
    )
    optimizer.solve(solver_name='glpk')
    expected_costs.append(optimizer.get_expected_cost())
    print(f"Penalty ${penalty:,}: Expected Cost = ${optimizer.get_expected_cost():,.2f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(penalties, expected_costs, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Shortage Penalty ($/unit)', fontsize=12)
ax.set_ylabel('Expected Total Cost ($)', fontsize=12)
ax.set_title('Sensitivity to Shortage Penalty', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()